Setting Up Pyspark

In [1]:
!apt-get update
!apt-get install openjdk-11-jdk -y
!pip install pyspark

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,793 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [4,126 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2

In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/usr/local/lib/python3.11/dist-packages/pyspark"

# Working with PySpark

We need to get or create a spark session

# Set a Spark session

The SparkSession is the entry point for high-level Spark functionality.
it is initiated using the `SparkSession.builder`

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import DateType

In [4]:
spk = SparkSession.builder.master("local[*]").appName('globalSales').getOrCreate()

In [5]:
print(spk.version)

3.5.5


In [8]:
global_df = spk.read.csv("global_sales_records.csv",header=True,inferSchema=True)
global_df.show(5)

+--------------------+--------------------+-------------+-------------+--------------+----------+---------+----------+----------+----------+---------+-------------+----------+------------+
|              Region|             Country|    Item Type|Sales Channel|Order Priority|Order Date| Order ID| Ship Date|Units Sold|Unit Price|Unit Cost|Total Revenue|Total Cost|Total Profit|
+--------------------+--------------------+-------------+-------------+--------------+----------+---------+----------+----------+----------+---------+-------------+----------+------------+
|Middle East and N...|          Azerbaijan|       Snacks|       Online|             C| 10/8/2014|535113847|10/23/2014|       934|    152.58|    97.44|    142509.72|  91008.96|    51500.76|
|Central America a...|              Panama|    Cosmetics|      Offline|             L| 2/22/2015|874708545| 2/27/2015|      4551|     437.2|   263.33|    1989697.2|1198414.83|   791282.37|
|  Sub-Saharan Africa|Sao Tome and Prin...|       Fruit

In [9]:
# get more information about the dataset
print(f"Dataset Shape\nRows: {global_df.count():,}\nColumns :{len(global_df.columns)}")

Dataset Shape
Rows: 100,000
Columns :14


In [10]:
global_df.printSchema() #data schema

root
 |-- Region: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Item Type: string (nullable = true)
 |-- Sales Channel: string (nullable = true)
 |-- Order Priority: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Order ID: integer (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Units Sold: integer (nullable = true)
 |-- Unit Price: double (nullable = true)
 |-- Unit Cost: double (nullable = true)
 |-- Total Revenue: double (nullable = true)
 |-- Total Cost: double (nullable = true)
 |-- Total Profit: double (nullable = true)



In [11]:
# Rename the columns, to remove the spaces
def rename_columns_remove_spaces(df: DataFrame) -> DataFrame:
    """Renames columns in a PySpark DataFrame to replace spaces with underscores."""
    for col_name in df.columns:
        new_col_name = col_name.replace(" ", "_")
        df = df.withColumnRenamed(col_name, new_col_name) # to rename column in pyspark
    return df

global_df = rename_columns_remove_spaces(global_df)

In [12]:
global_df.show(5)

+--------------------+--------------------+-------------+-------------+--------------+----------+---------+----------+----------+----------+---------+-------------+----------+------------+
|              Region|             Country|    Item_Type|Sales_Channel|Order_Priority|Order_Date| Order_ID| Ship_Date|Units_Sold|Unit_Price|Unit_Cost|Total_Revenue|Total_Cost|Total_Profit|
+--------------------+--------------------+-------------+-------------+--------------+----------+---------+----------+----------+----------+---------+-------------+----------+------------+
|Middle East and N...|          Azerbaijan|       Snacks|       Online|             C| 10/8/2014|535113847|10/23/2014|       934|    152.58|    97.44|    142509.72|  91008.96|    51500.76|
|Central America a...|              Panama|    Cosmetics|      Offline|             L| 2/22/2015|874708545| 2/27/2015|      4551|     437.2|   263.33|    1989697.2|1198414.83|   791282.37|
|  Sub-Saharan Africa|Sao Tome and Prin...|       Fruit

In [13]:
# Get null values if any
null_counts = global_df.select([F.count(F.when(F.isnan(c) | F.isnull(c), c)).alias(c) for c in global_df.columns])
null_counts.show()

+------+-------+---------+-------------+--------------+----------+--------+---------+----------+----------+---------+-------------+----------+------------+
|Region|Country|Item_Type|Sales_Channel|Order_Priority|Order_Date|Order_ID|Ship_Date|Units_Sold|Unit_Price|Unit_Cost|Total_Revenue|Total_Cost|Total_Profit|
+------+-------+---------+-------------+--------------+----------+--------+---------+----------+----------+---------+-------------+----------+------------+
|     0|      0|        0|            0|             0|         0|       0|        0|         0|         0|        0|            0|         0|           0|
+------+-------+---------+-------------+--------------+----------+--------+---------+----------+----------+---------+-------------+----------+------------+



# Data Transformation

In [15]:
# Transform the Order and Ship date to Date Type
global_df = global_df.withColumn("Order_Date",F.to_date(F.col("Order_Date"),"M/d/yyyy"))

In [16]:
global_df.show(5)

+--------------------+--------------------+-------------+-------------+--------------+----------+---------+----------+----------+----------+---------+-------------+----------+------------+
|              Region|             Country|    Item_Type|Sales_Channel|Order_Priority|Order_Date| Order_ID| Ship_Date|Units_Sold|Unit_Price|Unit_Cost|Total_Revenue|Total_Cost|Total_Profit|
+--------------------+--------------------+-------------+-------------+--------------+----------+---------+----------+----------+----------+---------+-------------+----------+------------+
|Middle East and N...|          Azerbaijan|       Snacks|       Online|             C|2014-10-08|535113847|10/23/2014|       934|    152.58|    97.44|    142509.72|  91008.96|    51500.76|
|Central America a...|              Panama|    Cosmetics|      Offline|             L|2015-02-22|874708545| 2/27/2015|      4551|     437.2|   263.33|    1989697.2|1198414.83|   791282.37|
|  Sub-Saharan Africa|Sao Tome and Prin...|       Fruit

In [20]:
# To Extract Year
global_df = global_df.withColumn("Order_Year", F.year("Order_Date"))

# To Extract Month
global_df = global_df.withColumn("Order_Month", F.date_format("Order_Date", "MMMM"))

# To Extract Days
global_df = global_df.withColumn("Order_Day", F.date_format("Order_Date", "E"))

# To Extract Quarter
global_df = global_df.withColumn("Order_Quarter", F.quarter("Order_Date"))

# To get the difference in days from order to shipping
global_df = global_df.withColumn("Shipping_Time_days", F.datediff("Ship_Date", "Order_Date"))

# Analysis

#  Insight 1. Top performing regions by revenue

In [18]:
region_perf_revenue = global_df.groupBy("Region").agg(F.sum("Total_Revenue").alias("Revenue_Generated")).orderBy(F.desc("Revenue_Generated"))
region_perf_revenue = region_perf_revenue.withColumn("Revenue_Generated", F.format_number(F.col("Revenue_Generated"), 2))
region_perf_revenue.show()

+--------------------+-----------------+
|              Region|Revenue_Generated|
+--------------------+-----------------+
|  Sub-Saharan Africa|34,958,453,406.17|
|              Europe|34,241,150,923.39|
|                Asia|19,293,401,219.82|
|Middle East and N...|16,921,412,794.52|
|Central America a...|14,553,730,165.29|
|Australia and Oce...|10,701,522,223.73|
|       North America| 2,937,002,333.49|
+--------------------+-----------------+



In [19]:
rg_pd = region_perf_revenue.toPandas()
rg_pd

,Region,Revenue_Generated
0,Sub-Saharan Africa,"34,958,453,406.17"
1,Europe,"34,241,150,923.39"
2,Asia,"19,293,401,219.82"
3,Middle East and North Africa,"16,921,412,794.52"
4,Central America and the Caribbean,"14,553,730,165.29"
5,Australia and Oceania,"10,701,522,223.73"
6,North America,"2,937,002,333.49"


# Insight 2. Overall Sales Trends and Seasonality

In [29]:
seasonal_sales = global_df.groupBy("Order_Year", "Order_Month").agg(F.sum("Total_Revenue").alias("Monthly_Revenue")).orderBy("Order_Year", "Order_Month")

seasonal_sales = seasonal_sales.withColumn("Monthly_Revenue",F.format_number(F.col("Monthly_Revenue"), 2))

seasonal_sales.show()

+----------+-----------+----------------+
|Order_Year|Order_Month| Monthly_Revenue|
+----------+-----------+----------------+
|      2010|      April|1,493,974,690.37|
|      2010|     August|1,607,025,923.59|
|      2010|   December|1,406,262,507.72|
|      2010|   February|1,351,908,048.52|
|      2010|    January|1,431,090,213.32|
|      2010|       July|1,462,505,704.06|
|      2010|       June|1,420,838,725.71|
|      2010|      March|1,386,193,257.95|
|      2010|        May|1,403,578,097.44|
|      2010|   November|1,468,104,337.15|
|      2010|    October|1,527,230,757.19|
|      2010|  September|1,571,013,876.09|
|      2011|      April|1,516,338,898.42|
|      2011|     August|1,385,406,678.42|
|      2011|   December|1,496,017,055.75|
|      2011|   February|1,268,096,111.48|
|      2011|    January|1,524,874,179.31|
|      2011|       July|1,492,257,151.44|
|      2011|       June|1,388,309,274.86|
|      2011|      March|1,521,722,825.51|
+----------+-----------+----------

In [35]:
snsales_pd = seasonal_sales.toPandas()
snsales_pd.head(10)

,Order_Year,Order_Month,Monthly_Revenue
0,2010,April,"1,493,974,690.37"
1,2010,August,"1,607,025,923.59"
2,2010,December,"1,406,262,507.72"
3,2010,February,"1,351,908,048.52"
4,2010,January,"1,431,090,213.32"
5,2010,July,"1,462,505,704.06"
6,2010,June,"1,420,838,725.71"
7,2010,March,"1,386,193,257.95"
8,2010,May,"1,403,578,097.44"
9,2010,November,"1,468,104,337.15"


#Insight 3. Underperforming Countries by Profit

In [31]:
upf_countries = global_df.groupBy("Country").agg(F.sum("Total_Profit").alias("Total_Profit")).orderBy("Total_Profit")
upf_countries = upf_countries.withColumn("Total_Profit",F.format_number(F.col("Total_Profit"), 2))
upf_countries.show()

+--------------------+--------------+
|             Country|  Total_Profit|
+--------------------+--------------+
|              Angola|178,155,609.15|
|     Solomon Islands|185,828,653.42|
|               Palau|185,972,902.81|
|          Kazakhstan|187,443,367.34|
|             Germany|188,026,206.60|
|         Switzerland|188,855,603.18|
|            Kiribati|189,363,092.17|
|             Nigeria|189,724,068.82|
|        Sierra Leone|190,921,135.72|
|             Jamaica|191,539,645.86|
|United States of ...|192,749,977.24|
|             Georgia|193,276,418.20|
|             Vietnam|194,177,410.45|
|          Kyrgyzstan|194,211,180.74|
|             Andorra|195,942,934.91|
|         Netherlands|196,098,273.33|
| Antigua and Barbuda|196,563,869.12|
|             Iceland|196,844,401.69|
|               Italy|197,617,559.77|
|             Moldova|197,954,823.08|
+--------------------+--------------+
only showing top 20 rows



In [39]:
upf_pd = upf_countries.toPandas()
upf_pd.head(15)

,Country,Total_Profit
0,Angola,"178,155,609.15"
1,Solomon Islands,"185,828,653.42"
2,Palau,"185,972,902.81"
3,Kazakhstan,"187,443,367.34"
4,Germany,"188,026,206.60"
5,Switzerland,"188,855,603.18"
6,Kiribati,"189,363,092.17"
7,Nigeria,"189,724,068.82"
8,Sierra Leone,"190,921,135.72"
9,Jamaica,"191,539,645.86"


#Insight 4. Most and Least Profitable Products

In [41]:
most_profitable = global_df.groupBy("Item_Type").agg(F.sum("Total_Profit").alias("Total_Profit")).orderBy(F.desc("Total_Profit"))
most_profitable = most_profitable.withColumn("Total_Profit",F.format_number(F.col("Total_Profit"), 2))
most_profitable.show()

+---------------+----------------+
|      Item_Type|    Total_Profit|
+---------------+----------------+
|      Cosmetics|7,289,406,555.68|
|      Household|6,870,966,095.35|
|Office Supplies|5,339,532,912.50|
|      Baby Food|4,017,647,893.20|
|         Cereal|3,743,318,890.62|
|        Clothes|3,067,841,433.60|
|     Vegetables|2,604,417,796.68|
|           Meat|2,387,834,992.40|
|         Snacks|2,299,287,932.88|
|  Personal Care|1,040,435,215.96|
|      Beverages|  650,112,575.58|
|         Fruits|   98,321,435.16|
+---------------+----------------+



In [42]:
mlp_pd = most_profitable.toPandas()
mlp_pd

,Item_Type,Total_Profit
0,Cosmetics,"7,289,406,555.68"
1,Household,"6,870,966,095.35"
2,Office Supplies,"5,339,532,912.50"
3,Baby Food,"4,017,647,893.20"
4,Cereal,"3,743,318,890.62"
5,Clothes,"3,067,841,433.60"
6,Vegetables,"2,604,417,796.68"
7,Meat,"2,387,834,992.40"
8,Snacks,"2,299,287,932.88"
9,Personal Care,"1,040,435,215.96"


#Insight 5. Sales Channel Comparison (Online vs. Offline)

In [44]:
channel_comp = global_df.groupBy("Sales_Channel").agg(F.sum("Total_Revenue").alias("Channel_Revenue"))
channel_comp = channel_comp.withColumn("Channel_Revenue",F.format_number(F.col("Channel_Revenue"), 2))
channel_comp.show()

+-------------+-----------------+
|Sales_Channel|  Channel_Revenue|
+-------------+-----------------+
|       Online|66,856,341,348.55|
|      Offline|66,750,331,717.86|
+-------------+-----------------+

